# GISAID Discrepancies

## Housekeeping

In [23]:
# Housekeeping

import os
import glob 
import pandas as pd
import xml.etree.ElementTree as ET
import requests
import time
import numpy as np
import dateutil 
from datetime import datetime
from collections import defaultdict 
import matplotlib.pyplot as plt

def partial_isolate(id):

    partial = id.replace("_", "-") # [-1] # If 25_, get the last bit
    partial = partial.replace("-original", "") # If -original suffix, remove
    partial = partial.replace("G", "-0") # If 25G, fix
    digits = partial.split("-")
    # Build partial isolates
    isolate = ""
    # other = ""
    for d in digits:
        # print(d)
        if len(d) == 2 and d.isnumeric(): # The 25 bit
            # isolate = d + "-"
            pass
        if len(d) == 6 and d.isnumeric(): # If it's just digits and not one of those weird isolates
            isolate = d + "-"
        if len(d) == 3 and d.isnumeric():
            isolate = isolate + d
        # elif d.isnumeric() == False: # If it's a weird isolate
        #     other = d + "-"
        # else: 
        #     other = partial 
    # Now add to list to check in Andersen files without doing wild for loops
    if len(isolate) == 10: # If this is a correctly formatted isolate
        # isolates.append(isolate)
        # All headers are followed by sequences
        partial_isolate = isolate
    else: # If this is some other isolate
        partial_isolate = partial

    return partial_isolate

In [24]:
os.chdir("C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/")

gisaid_feb_20_2026 = pd.read_excel("GISAID/downloads/2021-11-01--2026-02-20_Antarctica_North_America_South_America/gisaid_epiflu_isolates.xls")
gisaid_apr_14_2025 = pd.read_csv("GISAID_submission_dates_02-14-2022--12-26-2024_accessed_04-14-2025.csv")
genbank_feb_20_2026 = pd.read_csv("GenBank_submission_dates_02-14-2022--12-26-2024_accessed_02-20-2026.csv")
genbank_apr_14_2025 = pd.read_csv("GenBank_submission_dates_02-14-2022--12-26-2024_accessed_04-14-2025.csv")

In [25]:
gisaid_apr_14_2025["Submission_Date"] = gisaid_apr_14_2025["Submission_Date"].apply(lambda x: dateutil.parser.parse(x))
gisaid_apr_14_2025["Collection_Date"] = gisaid_apr_14_2025["Collection_Date"].apply(lambda x: dateutil.parser.parse(x))

gisaid_apr_14_2025 = gisaid_apr_14_2025[(gisaid_apr_14_2025["Submission_Date"] >= dateutil.parser.parse("01/01/2023")) & (gisaid_apr_14_2025["Submission_Date"] <= dateutil.parser.parse("09/17/2024"))]
gisaid_apr_14_2025["Isolate"] = gisaid_apr_14_2025["Isolate_Name"].apply(lambda x: x.split("/")[-2])
gisaid_apr_14_2025["Authors"] = gisaid_apr_14_2025["Authors"].apply(lambda x: x.split(", ")[0] if x==x else x)
print(gisaid_apr_14_2025.sort_values("Submission_Date")[["Submission_Date", "Isolate_Name", "Isolate"]])

     Submission_Date                               Isolate_Name        Isolate
991       2023-01-03        A/Striped Skunk/SK/FAV-0824-01/2022    FAV-0824-01
992       2023-01-03              A/Red Fox/SK/FAV-0824-96/2022    FAV-0824-96
993       2023-01-18      A/Canada goose/California/246038/2022         246038
994       2023-01-18     A/eared grebe/North Dakota/245625/2022         245625
995       2023-01-18  A/neotropic cormorant/Arizona/245467/2022         245467
...              ...                                        ...            ...
7368      2024-09-11       A/dairy cow/Idaho/24_024698-002/2024  24_024698-002
7367      2024-09-11       A/dairy cow/Colorado/024240-001/2024     024240-001
7371      2024-09-16    A/dairy cow/Michigan/24_014001-001/2024  24_014001-001
7370      2024-09-16    A/dairy cow/Michigan/24_014001-002/2024  24_014001-002
7369      2024-09-16    A/dairy cow/Michigan/24_014001-004/2024  24_014001-004

[6392 rows x 3 columns]


In [26]:
gisaid_feb_20_2026["Submission_Date"] = gisaid_feb_20_2026["Submission_Date"].apply(lambda x: dateutil.parser.parse(x))
gisaid_feb_20_2026["Collection_Date"] = gisaid_feb_20_2026["Collection_Date"].apply(lambda x: dateutil.parser.parse(x))

gisaid_feb_20_2026 = gisaid_feb_20_2026[(gisaid_feb_20_2026["Submission_Date"] >= dateutil.parser.parse("01/01/2023")) & (gisaid_feb_20_2026["Submission_Date"] <= dateutil.parser.parse("09/17/2024"))]
gisaid_feb_20_2026["Isolate"] = gisaid_feb_20_2026["Isolate_Name"].apply(lambda x: x.split("/")[-2])
# gisaid_feb_20_2026["Isolate_gisaid"] = gisaid_feb_20_2026["Isolate"]
# gisaid_feb_20_2026["Publication"]
gisaid_feb_20_2026["Geo_Location_Specific"] = gisaid_feb_20_2026["Location"].apply(lambda x: x.split(" / ")[-1] if "United States" in x else x.split(" / ")[1])
gisaid_feb_20_2026["Partial"] = gisaid_feb_20_2026["Isolate"].apply(partial_isolate)
print(gisaid_feb_20_2026.sort_values("Submission_Date")[["Submission_Date", "Isolate_Name"]])

      Submission_Date                               Isolate_Name
46         2023-01-03              A/Red Fox/SK/FAV-0824-96/2022
45         2023-01-03        A/Striped Skunk/SK/FAV-0824-01/2022
3530       2023-01-18  A/neotropic cormorant/Arizona/245467/2022
3529       2023-01-18     A/eared grebe/North Dakota/245625/2022
3528       2023-01-18      A/Canada goose/California/246038/2022
...               ...                                        ...
11173      2024-09-11       A/dairy cow/Colorado/024240-001/2024
1698       2024-09-13   A/buff-necked ibis/Bio bio/247636-1/2023
779        2024-09-16    A/dairy cow/Michigan/24_014001-001/2024
778        2024-09-16    A/dairy cow/Michigan/24_014001-002/2024
777        2024-09-16    A/dairy cow/Michigan/24_014001-004/2024

[6859 rows x 2 columns]


In [27]:
genbank_apr_14_2025["Submission_Date"] = genbank_apr_14_2025["Release_Date"].apply(lambda x: dateutil.parser.parse(x))
genbank_apr_14_2025["Collection_Date"] = genbank_apr_14_2025["Collection_Date"].apply(lambda x: dateutil.parser.parse(x))

genbank_apr_14_2025 = genbank_apr_14_2025[(genbank_apr_14_2025["Submission_Date"] >= dateutil.parser.parse("01/01/2023")) & (genbank_apr_14_2025["Submission_Date"] <= dateutil.parser.parse("09/17/2024"))]
genbank_apr_14_2025["Isolate_Name"] = genbank_apr_14_2025["GenBank_Title"].apply(lambda x: x.split("(")[1])
genbank_apr_14_2025["Isolate"] = genbank_apr_14_2025["Isolate_Name"].apply(lambda x: x.split("/")[-2])
# genbank_feb_20_2026["Isolate_genbank"] = genbank_feb_20_2026["Isolate"]
genbank_apr_14_2025["Authors"] = genbank_apr_14_2025["Submitters"].apply(lambda x: x.split(" (")[0])
print(genbank_apr_14_2025.sort_values("Submission_Date")[["Submission_Date", "Isolate_Name"]])

     Submission_Date                                       Isolate_Name
24        2023-02-04                      A/gray gull/Chile/C61947/2022
45        2023-02-04                      A/gray gull/Chile/C61947/2022
44        2023-02-04                      A/gray gull/Chile/C61947/2022
42        2023-02-04                      A/gray gull/Chile/C61947/2022
41        2023-02-04                      A/gray gull/Chile/C61947/2022
...              ...                                                ...
2748      2024-09-17  A/slender-billed parakeet/Araucania /242881-1/...
2749      2024-09-17  A/slender-billed parakeet/Araucania /242881-1/...
2750      2024-09-17  A/slender-billed parakeet/Araucania /242881-1/...
2751      2024-09-17  A/slender-billed parakeet/Araucania /242881-1/...
2753      2024-09-17  A/slender-billed parakeet/Araucania /242881-1/...

[2730 rows x 2 columns]


In [28]:
genbank_feb_20_2026["Submission_Date"] = genbank_feb_20_2026["Release_Date"].apply(lambda x: dateutil.parser.parse(x))
genbank_feb_20_2026["Collection_Date"] = genbank_feb_20_2026["Collection_Date"].apply(lambda x: dateutil.parser.parse(x))

genbank_feb_20_2026 = genbank_feb_20_2026[(genbank_feb_20_2026["Submission_Date"] >= dateutil.parser.parse("01/01/2023")) & (genbank_feb_20_2026["Submission_Date"] <= dateutil.parser.parse("09/17/2024"))]
genbank_feb_20_2026["Isolate_Name"] = genbank_feb_20_2026["GenBank_Title"].apply(lambda x: x.split("(")[1])
genbank_feb_20_2026["Isolate"] = genbank_feb_20_2026["Isolate_Name"].apply(lambda x: x.split("/")[-2])
# genbank_feb_20_2026["Isolate_genbank"] = genbank_feb_20_2026["Isolate"]
# genbank_feb_20_2026["Authors"] = genbank_feb_20_2026["Submitters"] #.apply(lambda x: x.split(" (")[0])
genbank_feb_20_2026["Partial"] = genbank_feb_20_2026["Isolate"].apply(partial_isolate)
genbank_feb_20_2026["Geo_Location_Specific"] = genbank_feb_20_2026["Geo_Location"].apply(lambda x: x.split(": ")[-1] if "USA" not in x else genbank_feb_20_2026[genbank_feb_20_2026["Geo_Location"] == x]["USA"].values[0])
print(genbank_feb_20_2026.sort_values("Submission_Date")[["Submission_Date", "Isolate_Name"]])

      Submission_Date                                       Isolate_Name
2069       2023-02-04                      A/gray gull/Chile/C61947/2022
2090       2023-02-04                      A/gray gull/Chile/C61947/2022
2089       2023-02-04                      A/gray gull/Chile/C61947/2022
2088       2023-02-04                      A/gray gull/Chile/C61947/2022
2086       2023-02-04                      A/gray gull/Chile/C61947/2022
...               ...                                                ...
26685      2024-09-17  A/slender-billed parakeet/Araucania /242881-1/...
26686      2024-09-17  A/slender-billed parakeet/Araucania /242881-1/...
26687      2024-09-17  A/slender-billed parakeet/Araucania /242881-1/...
26688      2024-09-17  A/slender-billed parakeet/Araucania /242881-1/...
26690      2024-09-17  A/slender-billed parakeet/Araucania /242881-1/...

[24622 rows x 2 columns]


In [29]:
genbank_gisaid_common = genbank_feb_20_2026.merge(gisaid_feb_20_2026, on="Isolate", suffixes=["_genbank", "_gisaid"])
print(genbank_gisaid_common)

       Accession GenBank_RefSeq         Assembly SRA_Accession BioSample  \
0     OQ352538.1        GenBank  GCA_039344385.1           NaN       NaN   
1     OQ352545.1        GenBank  GCA_039343775.1           NaN       NaN   
2     OQ352546.1        GenBank  GCA_039343775.1           NaN       NaN   
3     OQ352547.1        GenBank  GCA_039343775.1           NaN       NaN   
4     OQ352548.1        GenBank  GCA_039343775.1           NaN       NaN   
...          ...            ...              ...           ...       ...   
4950  PQ325300.1        GenBank  GCA_053356475.1           NaN       NaN   
4951  PQ327626.1        GenBank              NaN           NaN       NaN   
4952  PQ327627.1        GenBank              NaN           NaN       NaN   
4953  PQ327628.1        GenBank              NaN           NaN       NaN   
4954  PQ327629.1        GenBank              NaN           NaN       NaN   

        BioProject      Organism_Name                         Species  \
0      PRJNA80

In [30]:
genbank_gisaid_uncommon = genbank_feb_20_2026.merge(gisaid_feb_20_2026, on=["Collection_Date", "Submission_Date", "Partial"], suffixes=["_genbank", "_gisaid"])


In [31]:
gisaid_time = gisaid_apr_14_2025.merge(gisaid_feb_20_2026, on="Isolate_Id", suffixes=["_old", "_new"])
print(gisaid_time)

            Isolate_Id                                 PB2 Segment_Id_old  \
0     EPI_ISL_16367971     EPI2274963|A/Striped Skunk/SK/FAV-0824-01/2022   
1     EPI_ISL_16367899           EPI2274955|A/Red Fox/SK/FAV-0824-96/2022   
2     EPI_ISL_16555205  EPI2298226|PB2_A/Canada goose/California/22-02...   
3     EPI_ISL_16555204  EPI2298218|PB2_A/eared grebe/North Dakota/22-0...   
4     EPI_ISL_16555203  EPI2298210|PB2_A/neotropic cormorant/Arizona/2...   
...                ...                                                ...   
6380  EPI_ISL_19592596  EPI3675337|A/Glaucous-winged gull/Washington/W...   
6381  EPI_ISL_19592595  EPI3675329|A/Glaucous-winged gull/Washington/W...   
6382  EPI_ISL_19592618  EPI3675369|A/Harbor seal/Washington/W232510072...   
6383  EPI_ISL_19592610  EPI3675361|A/Harbor seal/Washington/W232490069...   
6384  EPI_ISL_19592604  EPI3675353|A/Harbor seal/Washington/W232430067...   

                                     PB1 Segment_Id_old  \
0        EPI2274

## Differences between GenBank and GISAID

### Isolates

All of the following have the same isolate "root", but different complete isolates. 

Isolate discrepancy examples:
- Sometimes the beginning hyphen in an isolate will be changed to an underscore (24-014001-004 -> 24_014001-004)
- Sometimes an appended "-original" at the end of the isolate will disappear (24-014001-004-original -> 24-014001-004)
- Sometimes the last two digits of a year at the beginning of an isolate will disappear (24-009038-001 -> 009038-001)
- Sometimes a repeated isolate will have the "-repeat2" appendage changed to "-R2" (24-015840-001-original-repeat2 -> 24_015840-001-R2)
- Sometimes, despite the same authors, collection date, location, and host, the isolate IDs are completely different (CEIRR-293 -> TN22-293)

In [32]:
print(genbank_gisaid_uncommon[genbank_gisaid_uncommon["Isolate_genbank"] != genbank_gisaid_uncommon["Isolate_gisaid"]][["Isolate_genbank", "Isolate_gisaid", "Isolate_Name_genbank", "Isolate_Name_gisaid"]])

            Isolate_genbank Isolate_gisaid  \
0             24-009038-001     009038-001   
1             24-009038-001     009038-001   
2             24-009038-001     009038-001   
3             24-009038-001     009038-001   
4             24-009038-001     009038-001   
..                      ...            ...   
323  24-014001-004-original  24_014001-004   
324  24-014001-004-original  24_014001-004   
325  24-014001-004-original  24_014001-004   
326  24-014001-004-original  24_014001-004   
327  24-014001-004-original  24_014001-004   

                          Isolate_Name_genbank  \
0    A/domestic cat/Montana/24-009038-001/2024   
1    A/domestic cat/Montana/24-009038-001/2024   
2    A/domestic cat/Montana/24-009038-001/2024   
3    A/domestic cat/Montana/24-009038-001/2024   
4    A/domestic cat/Montana/24-009038-001/2024   
..                                         ...   
323    A/cattle/MI/24-014001-004-original/2024   
324    A/cattle/MI/24-014001-004-original/2024 

In [33]:
print(genbank_gisaid_uncommon[genbank_gisaid_uncommon["Isolate_gisaid"] != genbank_gisaid_uncommon["Isolate_genbank"]][["Isolate_genbank", "Isolate_gisaid", "Isolate_Name_genbank", "Isolate_Name_gisaid"]][:50])

                    Isolate_genbank    Isolate_gisaid  \
0                     24-009038-001        009038-001   
1                     24-009038-001        009038-001   
2                     24-009038-001        009038-001   
3                     24-009038-001        009038-001   
4                     24-009038-001        009038-001   
5                     24-009038-001        009038-001   
6                     24-009038-001        009038-001   
7                     24-009038-001        009038-001   
8                     24-007379-002        007379-002   
9                     24-007379-002        007379-002   
10                    24-007379-002        007379-002   
11                    24-007379-002        007379-002   
12                    24-007379-002        007379-002   
13                    24-007379-002        007379-002   
14                    24-007379-002        007379-002   
15                    24-007379-002        007379-002   
16                    24-008552

### Isolate Names

All of these share isolate IDs, but have different full isolate names. This occurs in about half (2719/4955) of our sample. 

Isolate name discrepancy examples:
- GISAID sometimes cuts off or changes the specific type of animal to a general animal (snow goose -> goose, black vulture -> vulture, mallard -> duck, etc.)
- GISAID sometimes subtly changes the location name (Bio Bio -> Biobio, CHL -> Chile, etc.)
- GISAID sometimes changes the specific location to a more general location (Antofagasta -> Chile, etc.)
- Sometimes the isolate ids between genbank and gisaid are the same, but the isolate names are completely different (A/Missouri/121/2024 has the same isolate id as A/red-shouldered hawk/North Carolina/121/2022, etc.)

In [34]:
genbank_gisaid_common[genbank_gisaid_common["Isolate_Name_genbank"] != genbank_gisaid_common["Isolate_Name_gisaid"]][["Isolate_Name_genbank", "Isolate_Name_gisaid"]]

,Isolate_Name_genbank,Isolate_Name_gisaid
16,A/Gull/CHL/227023-3/2022,A/Gull/Chile/227023-3/2022
17,A/Gull/CHL/227023-3/2022,A/Gull/Chile/227023-3/2022
18,A/Gull/CHL/227023-3/2022,A/Gull/Chile/227023-3/2022
19,A/Gull/CHL/227023-2/2022,A/Gull/Chile/227023-2/2022
20,A/Gull/CHL/227023-2/2022,A/Gull/Chile/227023-2/2022
...,...,...
4942,A/gull/Bio Bio/237012/2023,A/gull/Biobio/237012/2023
4951,A/Missouri/121/2024,A/red-shouldered hawk/North Carolina/121/2022
4952,A/Missouri/121/2024,A/red-shouldered hawk/North Carolina/121/2022
4953,A/Missouri/121/2024,A/red-shouldered hawk/North Carolina/121/2022


### Collection Dates

All of these share isolate IDs, but have different collection dates in the metadata. This occurs in (117/4955) entries of our sample.

Date discrepancy examples:
- GISAID sometimes changes the collection date to the beginning of the month (2022-12-26 -> 12-01-2022, etc.)
- GISAID sometimes changes the collection date to a completely different date (2024-03-08 -> 2024-02-26, etc.)
- GISAID sometimes changes the collection date, if unknown and year 2025-2026, to the first of the year (2025 -> 2025-01-01, etc.) (Not in this sample)

In [35]:
genbank_gisaid_common[genbank_gisaid_common["Collection_Date_gisaid"] != genbank_gisaid_common["Collection_Date_genbank"]][["Isolate_Name_genbank", "Isolate_Name_gisaid", "Isolate", "Collection_Date_genbank", "Collection_Date_gisaid"]]

,Isolate_Name_genbank,Isolate_Name_gisaid,Isolate,Collection_Date_genbank,Collection_Date_gisaid
33,A/Pelecanus/Peru/VFAR-140/2022,A/Pelecanus/Peru/VFAR-140/2022,VFAR-140,2022-12-27,2022-12-01
35,A/Pelecanus/Peru/VFAR-140/2022,A/Pelecanus/Peru/VFAR-140/2022,VFAR-140,2022-12-27,2022-12-01
37,A/Pelecanus/Peru/VFAR-140/2022,A/Pelecanus/Peru/VFAR-140/2022,VFAR-140,2022-12-27,2022-12-01
39,A/Pelecanus/Peru/VFAR-140/2022,A/Pelecanus/Peru/VFAR-140/2022,VFAR-140,2022-12-27,2022-12-01
41,A/Pelecanus/Peru/VFAR-140/2022,A/Pelecanus/Peru/VFAR-140/2022,VFAR-140,2022-12-27,2022-12-01
...,...,...,...,...,...
4652,A/Brown Skua/South Georgia and the South Sandw...,A/dairy cow/Kansas/5/2024,5,2023-10-08,2024-04-27
4951,A/Missouri/121/2024,A/red-shouldered hawk/North Carolina/121/2022,121,2024-08-22,2022-02-12
4952,A/Missouri/121/2024,A/red-shouldered hawk/North Carolina/121/2022,121,2024-08-22,2022-02-12
4953,A/Missouri/121/2024,A/red-shouldered hawk/North Carolina/121/2022,121,2024-08-22,2022-02-12


### Host Names

All of these share isolate IDs, but have different host names in the metadata. This occurs in the **majority** (4811/4955) of our sample, though much of this can be explained by GISAID opting for the common name over the scientific name of each species.

Host name discrepancy examples:
- GISAID often just has the word "Host" in place of a host name (Leucophaeus modestus/gray gull -> Host)
- GISAID often, instead of using the specific species, has the host type in place of a host name (Theristicus caudatus/buff-necked ibis -> Avian)

In [36]:
genbank_gisaid_common[genbank_gisaid_common["Host_gisaid"] != genbank_gisaid_common["Host_genbank"]][["Isolate_Name_genbank", "Isolate_Name_gisaid", "Isolate", "Host_genbank", "Host_gisaid"]]

,Isolate_Name_genbank,Isolate_Name_gisaid,Isolate,Host_genbank,Host_gisaid
0,A/gray gull/Chile/C61947/2022,A/gray gull/Chile/C61947/2022,C61947,Leucophaeus modestus,Host
9,A/gray gull/Chile/C61947/2022,A/gray gull/Chile/C61947/2022,C61947,Leucophaeus modestus,Host
10,A/gray gull/Chile/C61947/2022,A/gray gull/Chile/C61947/2022,C61947,Leucophaeus modestus,Host
11,A/gray gull/Chile/C61947/2022,A/gray gull/Chile/C61947/2022,C61947,Leucophaeus modestus,Host
12,A/gray gull/Chile/C61947/2022,A/gray gull/Chile/C61947/2022,C61947,Leucophaeus modestus,Host
...,...,...,...,...,...
4950,A/buff-necked ibis/Bio bio/247636-1/2023,A/buff-necked ibis/Bio bio/247636-1/2023,247636-1,Theristicus caudatus,Avian
4951,A/Missouri/121/2024,A/red-shouldered hawk/North Carolina/121/2022,121,Homo sapiens,Buteo lineatus
4952,A/Missouri/121/2024,A/red-shouldered hawk/North Carolina/121/2022,121,Homo sapiens,Buteo lineatus
4953,A/Missouri/121/2024,A/red-shouldered hawk/North Carolina/121/2022,121,Homo sapiens,Buteo lineatus


### Locations

In [51]:
# print(genbank_gisaid_common[genbank_gisaid_common["Geo_Location_Specific_gisaid"] != genbank_gisaid_common["Geo_Location_Specific_genbank"]][["Isolate_Name_genbank", "Isolate_Name_gisaid", "Isolate", "Geo_Location_Specific_genbank", "Geo_Location_Specific_gisaid"]])
print(genbank_gisaid_common[(genbank_gisaid_common["Geo_Location_Specific_gisaid"] != genbank_gisaid_common["USA"]) & (genbank_gisaid_common["Country"] == "USA")][["Isolate_Name_genbank", "Isolate_Name_gisaid", "Isolate", "Geo_Location_Specific_genbank", "Geo_Location_Specific_gisaid"]][-50:])

                      Isolate_Name_genbank  \
4614                   A/Colorado/137/2024   
4615                   A/Colorado/137/2024   
4616                   A/Colorado/137/2024   
4617                   A/Colorado/137/2024   
4618                   A/Colorado/137/2024   
4619                   A/Colorado/137/2024   
4620                   A/Colorado/138/2024   
4621                   A/Colorado/138/2024   
4622                   A/Colorado/138/2024   
4623                   A/Colorado/138/2024   
4624                   A/Colorado/138/2024   
4625                   A/Colorado/138/2024   
4626                   A/Colorado/138/2024   
4627                   A/Colorado/138/2024   
4628                   A/Colorado/139/2024   
4629                   A/Colorado/139/2024   
4630                   A/Colorado/139/2024   
4631                   A/Colorado/139/2024   
4632                   A/Colorado/139/2024   
4633                   A/Colorado/139/2024   
4634                   A/Colorado/

## GISAID Metadata Updates

Sometimes, GISAID silently updates or changes metadata without warning.

GISAID metadata change examples:
- Sometimes GISAID appends or removes the last two digits of the year to the isolate ID, changing the strain name altogether (24_035820-001 -> 024710-001)
    - The last two digits of the year may be appended with an underscore, or a hyphen, making parsing the "root" isolate ID difficult (24-000001-001 vs. 24_000001-001)
    - Some isolate IDs include an "-original" appendage, making de-duplication between databases difficult (24-000001-001-original vs. 24-000001-001)
    - Some isolate IDs include an "-R2" or "-repeat2" appendage, making de-duplication between databases difficult (24-000001-001-R2 or 24-000001-001-original-repeat2 vs. 24-000001-001)
- Sometimes dates change completely without warning (2024-02-27 -> 2024-08-26)
- Sometimes dates are egregiously incorrect, both in the isolate name and the metadata, and must be updated later (A/bald eagle/Alaska/22-035250-002/1905, 1905-07-14 -> A/bald eagle/Alaska/22-035250-002/2022, 2022-02-27) 
- Genotypes may change without warning, or a genotype may be changed from an assigned to genotype to "Not assigned", even when the genotype is still current (B3.13 -> Notassigned)
    - Genotypes disappear in (135/6385) entries in our sample.
    - Rarely will a genotype be updated from "Not assigned" to an assigned genotype
- There is no mechanism in GISAID to know when metadata is updated -- meaning that all data must be downloaded each time just in case metadata (or the strain name altogether) has changed (A/dairy cow/USA/24_024710-001/2024 -> A/dairy cow/California/024710-001/2024)
    - While an "update date" column exists in the metadata, it mostly seems to only match the submission date -- otherwise, the update is either unclear or a removal of a previously assigned genotype without a replacement (B3.2 -> Notassigned)

In [38]:
print(gisaid_time[gisaid_time["Isolate_old"] != gisaid_time["Isolate_new"]][["Isolate_Name_old", "Isolate_Name_new", "Isolate_old", "Isolate_new"]])

                        Isolate_Name_old  \
6362  A/dairy cow/USA/24_024710-001/2024   
6363  A/dairy cow/USA/24_024710-006/2024   
6364  A/dairy cow/USA/24_024710-008/2024   
6365  A/dairy cow/USA/24_024710-009/2024   

                            Isolate_Name_new    Isolate_old Isolate_new  
6362  A/dairy cow/California/024710-001/2024  24_024710-001  024710-001  
6363  A/dairy cow/California/024710-006/2024  24_024710-006  024710-006  
6364  A/dairy cow/California/024710-008/2024  24_024710-008  024710-008  
6365  A/dairy cow/California/024710-009/2024  24_024710-009  024710-009  


In [39]:
print(gisaid_time[gisaid_time["Isolate_Name_old"] != gisaid_time["Isolate_Name_new"]][["Isolate_Name_old", "Isolate_Name_new", "Isolate_old", "Isolate_new", "Collection_Date_old", "Collection_Date_new"]])

                            Isolate_Name_old  \
4052  A/bald eagle/Alaska/22-035250-002/1905   
4053  A/bald eagle/Alaska/22-035250-001/1905   
6362      A/dairy cow/USA/24_024710-001/2024   
6363      A/dairy cow/USA/24_024710-006/2024   
6364      A/dairy cow/USA/24_024710-008/2024   
6365      A/dairy cow/USA/24_024710-009/2024   

                            Isolate_Name_new    Isolate_old    Isolate_new  \
4052  A/bald eagle/Alaska/22-035250-002/2022  22-035250-002  22-035250-002   
4053  A/bald eagle/Alaska/22-035250-001/2022  22-035250-001  22-035250-001   
6362  A/dairy cow/California/024710-001/2024  24_024710-001     024710-001   
6363  A/dairy cow/California/024710-006/2024  24_024710-006     024710-006   
6364  A/dairy cow/California/024710-008/2024  24_024710-008     024710-008   
6365  A/dairy cow/California/024710-009/2024  24_024710-009     024710-009   

     Collection_Date_old Collection_Date_new  
4052          1905-07-14          2022-02-27  
4053          1905-07-

In [40]:
print(gisaid_time[gisaid_time["Collection_Date_old"] != gisaid_time["Collection_Date_new"]][["Isolate_Name_old", "Isolate_Name_new", "Isolate_old", "Isolate_new", "Collection_Date_old", "Collection_Date_new"]])

                            Isolate_Name_old  \
4052  A/bald eagle/Alaska/22-035250-002/1905   
4053  A/bald eagle/Alaska/22-035250-001/1905   
6362      A/dairy cow/USA/24_024710-001/2024   
6363      A/dairy cow/USA/24_024710-006/2024   
6364      A/dairy cow/USA/24_024710-008/2024   
6365      A/dairy cow/USA/24_024710-009/2024   

                            Isolate_Name_new    Isolate_old    Isolate_new  \
4052  A/bald eagle/Alaska/22-035250-002/2022  22-035250-002  22-035250-002   
4053  A/bald eagle/Alaska/22-035250-001/2022  22-035250-001  22-035250-001   
6362  A/dairy cow/California/024710-001/2024  24_024710-001     024710-001   
6363  A/dairy cow/California/024710-006/2024  24_024710-006     024710-006   
6364  A/dairy cow/California/024710-008/2024  24_024710-008     024710-008   
6365  A/dairy cow/California/024710-009/2024  24_024710-009     024710-009   

     Collection_Date_old Collection_Date_new  
4052          1905-07-14          2022-02-27  
4053          1905-07-

In [41]:
print(gisaid_time[gisaid_time["Location_old"] != gisaid_time["Location_new"]][["Isolate_Name_old", "Isolate_Name_new", "Isolate_old", "Isolate_new","Location_old", "Location_new"]])

                        Isolate_Name_old  \
6362  A/dairy cow/USA/24_024710-001/2024   
6363  A/dairy cow/USA/24_024710-006/2024   
6364  A/dairy cow/USA/24_024710-008/2024   
6365  A/dairy cow/USA/24_024710-009/2024   

                            Isolate_Name_new    Isolate_old Isolate_new  \
6362  A/dairy cow/California/024710-001/2024  24_024710-001  024710-001   
6363  A/dairy cow/California/024710-006/2024  24_024710-006  024710-006   
6364  A/dairy cow/California/024710-008/2024  24_024710-008  024710-008   
6365  A/dairy cow/California/024710-009/2024  24_024710-009  024710-009   

                       Location_old  \
6362  North America / United States   
6363  North America / United States   
6364  North America / United States   
6365  North America / United States   

                                    Location_new  
6362  North America / United States / California  
6363  North America / United States / California  
6364  North America / United States / California  
636

In [42]:
print(gisaid_time[gisaid_time["Genotype_old"] != gisaid_time["Genotype_new"]][["Isolate_Name_old", "Isolate_Name_new", "Isolate_old", "Isolate_new","Genotype_old", "Genotype_new"]])
print(gisaid_time[(gisaid_time["Genotype_old"] != gisaid_time["Genotype_new"]) & (gisaid_time["Genotype_new"].str.contains("Notassigned"))][["Isolate_Name_old", "Isolate_Name_new", "Isolate_old", "Isolate_new","Genotype_old", "Genotype_new"]])

                                       Isolate_Name_old  \
3527             A/Great_Horned_Owl/BC/FAV-0856-18/2022   
3528       A/Great_Black-Backed_Gull/QC/FAV-0870-4/2022   
3529      A/Great_Black-Backed_Gull/QC/FAV-0829-12/2022   
3536             A/Northwestern_Crow/BC/FAV-0669-4/2022   
5236         A/chicken/Iowa/23-038644-001-original/2023   
...                                                 ...   
6343        A/mallard/Maine/24-004330-004-original/2024   
6345        A/mallard/Maine/24-004329-016-original/2024   
6346        A/mallard/Maine/24-004329-015-original/2024   
6347  A/green-winged teal/North Carolina/24-004327-0...   
6370               A/dairy cow/Colorado/024240-001/2024   

                                       Isolate_Name_new  \
3527             A/Great_Horned_Owl/BC/FAV-0856-18/2022   
3528       A/Great_Black-Backed_Gull/QC/FAV-0870-4/2022   
3529      A/Great_Black-Backed_Gull/QC/FAV-0829-12/2022   
3536             A/Northwestern_Crow/BC/FAV-0669-4/2022

In [43]:
print(gisaid_time[gisaid_time["Update_Date_old"] != gisaid_time["Update_Date_new"]][["Isolate_Name_old", "Isolate_Name_new", "Update_Date_old", "Update_Date_new"]]) # All NaN


                                       Isolate_Name_old  \
202               A/black vulture/Georgia/W22-719C/2022   
203              A/black vulture/Virginia/W22-662A/2022   
204                A/black vulture/Georgia/W22-395/2022   
205               A/black vulture/Georgia/W22-404B/2022   
206               A/black vulture/Georgia/W22-675A/2022   
...                                                 ...   
6380  A/Glaucous-winged gull/Washington/W232270041-2...   
6381  A/Glaucous-winged gull/Washington/W231990001-1...   
6382       A/Harbor seal/Washington/W232510072-3-1/2023   
6383         A/Harbor seal/Washington/W232490069-3/2023   
6384       A/Harbor seal/Washington/W232430067-2-3/2023   

                                       Isolate_Name_new Update_Date_old  \
202               A/black vulture/Georgia/W22-719C/2022             NaN   
203              A/black vulture/Virginia/W22-662A/2022             NaN   
204                A/black vulture/Georgia/W22-395/2022           

In [44]:
print(gisaid_time[(gisaid_time["Update_Date_old"] != gisaid_time["Submission_Date_old"]) & (gisaid_time["Update_Date_old"] == gisaid_time["Update_Date_old"])][["Isolate_Name_old", "Isolate_Name_new", "Update_Date_old", "Submission_Date_old", "Collection_Date_old", "Collection_Date_new", "Location_old", "Location_new", "Genotype_old", "Genotype_new", "Host_old", "Host_new"]])

                                       Isolate_Name_old  \
7                       A/gull/Maine/22-022526-005/2022   
9     A/great black-backed gull/Massachusetts/22-025...   
12    A/great black-backed gull/Maine/22-022526-015/...   
14    A/great black-backed gull/Maine/22-022526-007/...   
16    A/great black-backed gull/Massachusetts/22-016...   
...                                                 ...   
5228                        A/cat/South Dakota/001/2024   
5272                A/dairy cow/Ohio/24-010202-009/2024   
5651               A/dairy cow/Idaho/24-011573-007/2024   
5893     A/mallard/Missouri/23-034483-001-original/2023   
6005               A/dairy cow/Texas/24_009308-004/2024   

                                       Isolate_Name_new Update_Date_old  \
7                       A/gull/Maine/22-022526-005/2022      2023-01-27   
9     A/great black-backed gull/Massachusetts/22-025...      2023-01-27   
12    A/great black-backed gull/Maine/22-022526-015/...      2023-